# Relational Composition with Join

This notebook demonstrates how `Join` works as relational composition.

We use biblical genealogy data: a `Parent` relation (person → direct parent).
By composing it with itself we get grandparents, great-grandparents, and so on.
`Closure` iterates this to convergence — the full ancestor relation.

In [ ]:
import sys
import numpy as np
import pandas as pd
from core.tensor import Tensor

T = Tensor()

## Load data

We fetch the BradyStephenson bible dataset and build a binary `Parent` matrix.
`Parent[i, j] = 1` means person `i` is a direct parent of person `j`.

In [2]:
base_url   = "https://raw.githubusercontent.com/BradyStephenson/bible-data/main"
df_persons = pd.read_csv(f"{base_url}/BibleData-Person.csv")
df_rels    = pd.read_csv(f"{base_url}/BibleData-PersonRelationship.csv")

print(f"Loaded {len(df_persons)} people and {len(df_rels)} relationship records.")

Loaded 3009 people and 5450 relationship records.


In [3]:
# Keep only parent/child edges and normalise to parent → child direction
target_rels  = ['father', 'mother', 'son', 'daughter']
df_family    = df_rels[df_rels['relationship_type'].str.lower().isin(target_rels)].copy()
mask_parent  = df_family['relationship_type'].str.lower().isin(['father', 'mother'])

df_family['parent_id'] = np.where(mask_parent, df_family['person_id_1'], df_family['person_id_2'])
df_family['child_id']  = np.where(mask_parent, df_family['person_id_2'], df_family['person_id_1'])
df_family = df_family.dropna(subset=['parent_id', 'child_id'])

valid_ids  = sorted(set(df_family['parent_id']) | set(df_family['child_id']))
id_to_idx  = {rid: i for i, rid in enumerate(valid_ids)}
id_to_name = df_persons.set_index('person_id')['person_name']
names      = [id_to_name.get(rid, f"Unknown_{rid}") for rid in valid_ids]
N          = len(valid_ids)

Parent = np.zeros((N, N), dtype=float)
for _, row in df_family.iterrows():
    pi = id_to_idx[row['parent_id']]
    ci = id_to_idx[row['child_id']]
    Parent[pi, ci] = 1.0

print(f"Matrix shape:         {Parent.shape}")
print(f"Direct parent edges:  {int(Parent.sum())}")

Matrix shape:         (1972, 1972)
Direct parent edges:  1727


## Join as relational composition

`Join(A, B)[x, z] = max_y min(A[x,y], B[y,z])`

This is the lattice generalisation of boolean matrix multiplication.
At `temp=0` it is exact: there is an edge `(x,z)` iff there exists a `y`
with both `A[x,y]` and `B[y,z]` set.

Composing `Parent` with itself gives the **grandparent** relation.

In [4]:
Grandparent      = T.Join(Parent, Parent,      temp=0.0)
GreatGrandparent = T.Join(Parent, Grandparent, temp=0.0)

print(f"Direct parent edges:       {int(Parent.sum()):>6}")
print(f"Grandparent edges:         {int((Grandparent > 0).sum()):>6}")
print(f"Great-grandparent edges:   {int((GreatGrandparent > 0).sum()):>6}")

Direct parent edges:         1727
Grandparent edges:           1421
Great-grandparent edges:     1435


## Spot-check: Adam's children, grandchildren, great-grandchildren

Each additional `Join` step reaches one generation further.

In [5]:
def lookup(name_query):
    return next((i for i, n in enumerate(names) if name_query.lower() in n.lower()), None)

def show(matrix, idx, label):
    idxs = np.where(matrix[idx] > 0)[0]
    print(f"  {label} ({len(idxs)}): {[names[i] for i in idxs]}")

adam = lookup('Adam')
print(f"Adam = '{names[adam]}'")
show(Parent,           adam, 'direct children')
show(Grandparent,      adam, 'grandchildren')
show(GreatGrandparent, adam, 'great-grandchildren')

Adam = 'Adam'
  direct children (3): ['Abel', 'Cain', 'Seth']
  grandchildren (2): ['Enoch', 'Enosh']
  great-grandchildren (2): ['Irad', 'Kenan']


## Closure: iterating Join to convergence

`Closure` repeatedly applies `Join` until the relation stops growing.
The result is the **full ancestor** relation:
`Ancestor[i,j] = 1` iff `i` is an ancestor of `j` at any depth.

In [6]:
Ancestor = T.Closure(Parent, temp=0.0)

print(f"Direct parent edges:           {int(Parent.sum()):>6}")
print(f"Full ancestor edges (closure): {int((Ancestor > 0).sum()):>6}")
print(f"Growth factor:                 {(Ancestor > 0).sum() / Parent.sum():.1f}x")

✓ CONVERGED at iteration 75
Direct parent edges:             1727
Full ancestor edges (closure):  33944
Growth factor:                 19.7x


## Querying the ancestor relation

In [7]:
def descendants_of(name_query):
    idx  = lookup(name_query)
    idxs = np.where(Ancestor[idx] > 0)[0]
    print(f"{names[idx]} is ancestor of {len(idxs)} people.")
    print(f"  first 10: {[names[i] for i in idxs[:10]]}")

def ancestors_of(name_query):
    idx  = lookup(name_query)
    idxs = np.where(Ancestor[:, idx] > 0)[0]
    print(f"{names[idx]} is descended from {len(idxs)} people.")
    print(f"  ancestors: {[names[i] for i in idxs]}")

descendants_of('Adam')
print()
ancestors_of('Abram')

Adam is ancestor of 821 people.
  first 10: ['Aaron', 'Abdi', 'Abdon', 'Abel', 'Abiasaph', 'Abida', 'Abihail', 'Abihu', 'Abihud', 'Abijah']

Abram is descended from 20 people.
  ancestors: ['Adam', 'Arpachshad', 'Eber', 'Enoch', 'Enosh', 'Eve', 'Jared', 'Kenan', 'Lamech', 'Mahalalel', 'Methuselah', 'Nahor', 'Noah', 'Peleg', 'Reu', 'Serug', 'Seth', 'Shelah', 'Shem', 'Terah']


## Composing different relations

`Join` works across any compatible pair of relations, not just self-composition.
Here we build a `Sibling` relation (share a parent) by composing `Parent.T` with `Parent` —
"go up to a parent, then back down to a child" — and then compose that with `Ancestor`
to get the set of people who are descendants of any of Abram's siblings.

In [8]:
# Sibling[i,j] = 1 iff i and j share at least one parent
Sibling = T.Join(Parent.T, Parent, temp=0.0)
np.fill_diagonal(Sibling, 0)   # a person is not their own sibling

# Ancestor of a sibling: go sideways, then forward in time
AncestorOfSibling = T.Join(Sibling, Ancestor, temp=0.0)

print(f"Sibling edges:              {int((Sibling > 0).sum()):>6}")
print(f"Ancestor-of-sibling edges:  {int((AncestorOfSibling > 0).sum()):>6}")

abram = lookup('Abram')
idxs  = np.where(AncestorOfSibling[abram] > 0)[0]
print(f"\nDescendants of Abram's siblings ({len(idxs)} total):")
print(f"  first 10: {[names[i] for i in idxs[:10]]}")

Sibling edges:                4930
Ancestor-of-sibling edges:   46051

Descendants of Abram's siblings (689 total):
  first 10: ['Aaron', 'Abdi', 'Abdon', 'Abiasaph', 'Abihail', 'Abihu', 'Abihud', 'Abijah', 'Abijah', 'Abijam']
